# nanochat on Kaggle (GPU T4 x2, 複数セッション対応)

Kaggle の無料 GPU で nanochat のフルパイプライン（トークナイザ → 事前学習 → SFT → チャット）を回すノートブック。
12時間のセッションに収まらない分は、チェックポイントを Output に残して次のセッションで再開する。

**事前準備（右パネル → Settings）**
- Accelerator: **GPU T4 x2**（毎回同じ GPU 数にすること。optimizer の状態は GPU ごとに分割保存されている）
- Internet: **On**（電話番号認証が必要）

**セッションの回し方**
1. 1回目: そのまま「Save Version → Save & Run All (Commit)」。時間切れ前に自動で保存して終了する
2. 2回目以降: 「Add Input → Your Work → このノートブック」で**直前のバージョンの Output** を入力に追加して、再び Commit。前回のチェックポイントから自動で再開する
3. 事前学習が終わると、同じセッション内で時間が足りれば SFT まで進む（足りなければ次のセッションで SFT）

対話セッションの `/kaggle/working` はセッション終了で消えるので、またぐ場合は必ず Commit で実行する。

**T4 向けの主な変更点**
- T4 (SM75) は bf16 / Flash Attention 3 非対応 → `NANOCHAT_DTYPE=float16`（GradScaler 自動有効）＋ SDPA
- SDPA はスライディングウィンドウが遅いので `--window-pattern=L`
- コンテキスト長 1024、モデルは d8〜d12 程度

In [ ]:
import time
NB_T0 = time.time()  # セッション開始時刻（学習の時間予算の計算に使う）
!nvidia-smi

## 設定
`DEPTH` / `MAX_SEQ_LEN` / `DEVICE_BATCH_SIZE` / `NUM_SHARDS` は**セッション間で変えないこと**（学習率スケジュールやデータの読み位置がずれる）。

In [ ]:
import os, glob, json, shutil

REPO_URL = "https://github.com/OneShining06/nanochat_fork.git"  # private リポジトリならトークン付き URL にする
DEPTH = 8                 # 8: 1セッション前後 / 12: 複数セッション前提
MAX_SEQ_LEN = 1024
DEVICE_BATCH_SIZE = 8     # GPU あたりのマイクロバッチ。OOM したら 4 に。総バッチは勾配累積で自動的に揃う
NUM_SHARDS = 24           # 事前学習データのシャード数（1シャード ≈ 100MB）。d12 なら 40 程度。減らすのは不可
SESSION_HOURS = 12        # Kaggle の1セッションの上限
MARGIN_MIN = 45           # 終了処理（保存・コピー）用の余裕。12h を超えると Output が残らないので削らない
SFT_MIN_HOURS = 3         # 残り時間がこれ未満なら SFT は次のセッションに回す
WANDB_RUN = "kaggle-d8"   # "dummy" で wandb 無効。Add-ons → Secrets に WANDB_API_KEY を登録し、このノートブックで有効にしておく

BASE = "/tmp/nanochat"                  # データ/チェックポイント（/kaggle/working は 20GB 制限なので一時領域に置く）
OUT = "/kaggle/working/nanochat_out"    # 次のセッションに引き継ぐもの（Output として保存される）
REPO = "/tmp/nanochat_fork"             # コードは Output に含めない
MODEL_TAG = f"d{DEPTH}"

import torch
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])

os.environ["NANOCHAT_BASE_DIR"] = BASE
os.environ["NANOCHAT_DTYPE"] = "float16"            # T4 は bf16 非対応
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NCCL_P2P_DISABLE"] = "1"                # Kaggle の T4x2 で DDP がハングする事例への対策
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1 GPU なら python、2 GPU なら torchrun
LAUNCH = f"torchrun --standalone --nproc_per_node={NGPU}" if NGPU > 1 else "python"
SEP = "--" if NGPU > 1 else ""

def remaining_min():
    return SESSION_HOURS * 60 - MARGIN_MIN - (time.time() - NB_T0) / 60

## コード取得と依存関係のインストール
Kaggle にプリインストール済みの PyTorch をそのまま使う（`pyproject.toml` の torch==2.9.1 を入れ直すと数GBのダウンロードになるため）。

In [ ]:
if not os.path.exists(REPO):
    !git clone --depth 1 {REPO_URL} {REPO}
%cd {REPO}
!git log -1 --oneline
!pip install -q rustbpe tiktoken kernels filelock psutil pyarrow wandb pyyaml

In [ ]:
# 任意: wandb ログイン（Add-ons → Secrets に WANDB_API_KEY を登録しておく）
if WANDB_RUN != "dummy":
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 前回セッションの成果物を復元

In [ ]:
prev = sorted(glob.glob("/kaggle/input/**/nanochat_out", recursive=True))
if prev:
    if len(prev) > 1:
        print("WARNING: 前回の成果物が複数見つかった。先頭を使う:", prev)
    print("前回の成果物から復元:", prev[0])
    shutil.copytree(prev[0], BASE, dirs_exist_ok=True)
else:
    print("前回の成果物なし: 最初から学習する")

def base_state():
    """最新の事前学習チェックポイントの (step, 完了したか) を返す"""
    metas = sorted(glob.glob(f"{BASE}/base_checkpoints/{MODEL_TAG}/meta_*.json"))
    if not metas:
        return None, False
    meta = json.load(open(metas[-1]))
    return meta["step"], meta["step"] == meta.get("num_iterations")

def sft_done():
    return bool(glob.glob(f"{BASE}/chatsft_checkpoints/{MODEL_TAG}/model_*.pt"))

RESUME_STEP, BASE_DONE = base_state()
if RESUME_STEP is not None and not BASE_DONE:
    n_optim = len(glob.glob(f"{BASE}/base_checkpoints/{MODEL_TAG}/optim_{RESUME_STEP:06d}_rank*.pt"))
    assert n_optim == NGPU, f"optimizer は {n_optim} GPU 分で保存されている。同じ GPU 数で再開すること"
print(f"事前学習: step={RESUME_STEP}, 完了={BASE_DONE} | SFT 完了={sft_done()}")

## データ取得とトークナイザ学習
データは毎セッション再ダウンロードする（数分）。トークナイザは前回のものがあれば再学習しない（作り直すとモデルと合わなくなる）。

In [ ]:
!python -m nanochat.dataset -n {NUM_SHARDS}
if not os.path.isdir(f"{BASE}/tokenizer"):
    !python -m scripts.tok_train
    !python -m scripts.tok_eval

## 事前学習
セッションの残り時間を `--time-budget` に渡し、時間切れ前にチェックポイントを保存して終了する。

In [ ]:
if not BASE_DONE:
    budget = int(remaining_min())
    resume = f"--resume-from-step={RESUME_STEP}" if RESUME_STEP is not None else ""
    print(f"time budget: {budget} min {resume}")
    cmd = (f"{LAUNCH} -m scripts.base_train {SEP} --depth={DEPTH} --max-seq-len={MAX_SEQ_LEN} --window-pattern=L"
           f" --device-batch-size={DEVICE_BATCH_SIZE} --eval-every=250 --eval-tokens=2097152"
           f" --core-metric-every=-1 --sample-every=1000 --save-every=1000"
           f" --time-budget={budget} {resume} --run={WANDB_RUN}")
    !{cmd}
RESUME_STEP, BASE_DONE = base_state()
print(f"事前学習: step={RESUME_STEP}, 完了={BASE_DONE}")

## 評価と SFT（会話形式の学習）
事前学習が完了していて、残り時間が `SFT_MIN_HOURS` 以上あるときだけ実行する。

In [ ]:
if BASE_DONE and not sft_done() and remaining_min() > SFT_MIN_HOURS * 60:
    # 軽量な評価（CORE はタスクあたり 100 件に制限）
    !{LAUNCH} -m scripts.base_eval {SEP} --device-batch-size={DEVICE_BATCH_SIZE} --split-tokens=2097152 --max-per-task=100
    !{LAUNCH} -m scripts.chat_sft {SEP} --eval-every=200 --eval-tokens=2097152 --chatcore-every=-1 --run={WANDB_RUN}
    !{LAUNCH} -m scripts.chat_eval {SEP} -i sft -x 200
else:
    print(f"SFT はスキップ: 事前学習完了={BASE_DONE}, SFT 完了={sft_done()}, 残り {remaining_min():.0f} 分")

## チャットしてみる

In [ ]:
if sft_done():
    !python -m scripts.chat_cli -p "What is the capital of France?"
    !python -m scripts.chat_cli -p "Why is the sky blue?"

## 次のセッションへの引き継ぎ
トークナイザと**最新の**チェックポイントだけを `/kaggle/working/nanochat_out` にコピーする（学習データやコードは含めない）。
事前学習の再開には optimizer の状態も必要なので一緒にコピーする。

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
if os.path.isdir(f"{BASE}/tokenizer"):
    shutil.copytree(f"{BASE}/tokenizer", f"{OUT}/tokenizer")

def copy_latest(phase_dir, with_optim):
    src = f"{BASE}/{phase_dir}/{MODEL_TAG}"
    metas = sorted(glob.glob(f"{src}/meta_*.json"))
    if not metas:
        return
    step = json.load(open(metas[-1]))["step"]
    dst = f"{OUT}/{phase_dir}/{MODEL_TAG}"
    os.makedirs(dst, exist_ok=True)
    patterns = [f"model_{step:06d}.pt", f"meta_{step:06d}.json"] + ([f"optim_{step:06d}_rank*.pt"] if with_optim else [])
    for pattern in patterns:
        for f in glob.glob(f"{src}/{pattern}"):
            shutil.copy2(f, dst)

copy_latest("base_checkpoints", with_optim=not sft_done())  # SFT が済んだら再開用の optimizer は不要
copy_latest("chatsft_checkpoints", with_optim=False)
!du -sh {OUT}/* {OUT}/*/*
print(f"経過 {(time.time() - NB_T0) / 3600:.2f} h")